In [163]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
from sklearn.preprocessing import TargetEncoder
from sklearn.model_selection import train_test_split
import tensorflow as tf
from keras.layers import Dense
from keras import Sequential
from lazypredict.Supervised import LazyRegressor
from sklearn import metrics
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

In [164]:
df = pd.read_csv('bayut_listings_cleaned.csv')
# df = pd.read_csv('bayut_listings_cleaned_notDuplicated.csv')

y = df['Price']
X = df.drop('Price', axis=1)
df.head()

,Price,Property Type,Number Of Bedrooms,Number Of Bathrooms,Area,Location
0,1650000,FLOOR,5,4,523,OTHER
1,1249000,FLOOR,6,5,500,OTHER
2,499000,APARTMENT,3,3,154,OTHER
3,499000,APARTMENT,3,3,154,OTHER
4,499000,APARTMENT,3,3,154,OTHER


In [165]:
x_train,x_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [166]:
def targetEncoding(column: str, x: pd.DataFrame, y: pd.Series) -> pd.DataFrame:
    targetenc = TargetEncoder(smooth='auto', shuffle=False)
    x[column] = targetenc.fit_transform(x[[column]],y)
    return x



In [167]:
x_train = targetEncoding(column='Location', x= x_train, y= y_train)
x_test = targetEncoding(column='Location', x= x_test, y= y_test)

In [168]:
x_train = pd.get_dummies(x_train,columns=['Property Type'],dtype=int)
x_test = pd.get_dummies(x_test,columns=['Property Type'],dtype=int)

In [169]:
reg = LazyRegressor(verbose=0,ignore_warnings=False, custom_metric=None )
models,predictions = reg.fit(x_train, x_test, y_train, y_test)
models.head()

100%|██████████| 42/42 [00:29<00:00,  1.43it/s]

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000140 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 298
[LightGBM] [Info] Number of data points in the train set: 11113, number of used features: 8
[LightGBM] [Info] Start training from score 1406731.310987


,Adjusted R-Squared,R-Squared,RMSE,Time Taken
Model,,,,
ExtraTreesRegressor,0.33,0.33,1764259.05,0.56
GradientBoostingRegressor,0.29,0.29,1808580.04,0.30
RandomForestRegressor,0.29,0.29,1814709.66,0.87
KNeighborsRegressor,0.27,0.27,1833142.67,0.08
LGBMRegressor,0.27,0.27,1840712.95,0.04


In [170]:
models

,Adjusted R-Squared,R-Squared,RMSE,Time Taken
Model,,,,
ExtraTreesRegressor,0.33,0.33,1764259.05,0.56
GradientBoostingRegressor,0.29,0.29,1808580.04,0.30
RandomForestRegressor,0.29,0.29,1814709.66,0.87
KNeighborsRegressor,0.27,0.27,1833142.67,0.08
LGBMRegressor,0.27,0.27,1840712.95,0.04
HistGradientBoostingRegressor,0.27,0.27,1841228.75,0.06
BaggingRegressor,0.25,0.25,1862188.38,0.10
XGBRegressor,0.24,0.25,1869957.32,0.04
AdaBoostRegressor,0.24,0.24,1871352.49,0.07


In [ ]:
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import GridSearchCV

param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 5, 10, 20, 30],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4]
}

pipeline = Pipeline([
    # ('scalar', StandardScaler()),
     ('poly', PolynomialFeatures(degree=2)),
    ('model', RandomForestRegressor(random_state=42))
])

grid = GridSearchCV(pipeline, param_grid, cv=5, n_jobs=-1, verbose=1)
grid.fit(x_train, y_train)

print("Best parameters:", grid.best_params_)
print("Train score:", grid.score(x_train, y_train))
print("Test score:", grid.score(x_test, y_test))


Fitting 5 folds for each of 90 candidates, totalling 450 fits


In [172]:
param_grid = {
    'model__n_estimators': [200],
    'model__max_depth': [None],
    'model__min_samples_split': [2],
    'model__min_samples_leaf': [1],
}

pipeline = Pipeline([
    # ('scalar', StandardScaler()),
     ('poly', PolynomialFeatures(degree=2)),
    ('model', RandomForestRegressor())
])

grid = GridSearchCV(pipeline, param_grid, cv=5, n_jobs=-1, verbose=1,scoring='r2')
grid.fit(x_train, y_train)

print("Best parameters:", grid.best_params_)
print("Train score:", grid.score(x_train, y_train))
print("Test score:", grid.score(x_test, y_test))

Fitting 5 folds for each of 1 candidates, totalling 5 fits
Best parameters: {'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 200}
Train score: 0.8417683206403133
Test score: 0.33416568101900035


In [173]:
forest = RandomForestRegressor(random_state=42, n_estimators=200,max_depth=23,min_samples_leaf=4,min_samples_split=4).fit(x_train, y_train)
print("Train score:", forest.score(x_train, y_train))
print("Test score:", forest.score(x_test, y_test))

Train score: 0.6540867614021255
Test score: 0.2913734096126157


In [174]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import GridSearchCV
from lightgbm import LGBMRegressor

# Define parameter grid for LGBMRegressor
param_grid = {
    'model__n_estimators': [100],
    'model__max_depth': [-1],
    'model__learning_rate': [0.1],
    'model__num_leaves': [31],
    'model__min_child_samples': [5]
}

# Pipeline using LightGBM
pipeline = Pipeline([
    # ('scalar', StandardScaler()),  # Optional: may not be needed for tree-based models
    ('poly', PolynomialFeatures(degree=2)),
    ('model', LGBMRegressor(random_state=42))
])

# GridSearchCV with cross-validation
grid = GridSearchCV(pipeline, param_grid, cv=5, n_jobs=-1, verbose=1)
grid.fit(x_train, y_train)

# Output the results
print("Best parameters:", grid.best_params_)
print("Train score:", grid.score(x_train, y_train))
print("Test score:", grid.score(x_test, y_test))


Fitting 5 folds for each of 1 candidates, totalling 5 fits
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000216 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2118
[LightGBM] [Info] Number of data points in the train set: 11113, number of used features: 37
[LightGBM] [Info] Start training from score 1406731.310987
Best parameters: {'model__learning_rate': 0.1, 'model__max_depth': -1, 'model__min_child_samples': 5, 'model__n_estimators': 100, 'model__num_leaves': 31}
Train score: 0.7312106392947426
Test score: 0.2768978605039455


In [175]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(pipeline, x_train, y_train, cv=5)
print("Cross-validation scores:", scores)
print("Average CV score:", scores.mean())

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000436 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2074
[LightGBM] [Info] Number of data points in the train set: 8890, number of used features: 37
[LightGBM] [Info] Start training from score 1419124.895613
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000162 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2081
[LightGBM] [Info] Number of data points in the train set: 8890, number of used features: 37
[LightGBM] [Info] Start training from score 1404813.540270
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000146 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info